# chesscog_compare: pretrained chesscog vs ours on my board (M4)

chesscog (github.com/georg-wolflein/chesscog, Wolflein & Arandjelovic 2021) is a
comparison baseline only; nothing in the Gambit pipeline depends on it.

Workflow (driven by `run_chesscog.ipynb`):
1. `run_chesscog()` on `data/crops/<session>/frames/*.png` of the test sessions
   -> save a CSV with columns `frame_id, session_id, fen_chesscog`
2. our evaluate run writes per-board predictions (`frame_id, session_id, fen_true, fen_ours`)
3. `compare()` joins both into `results/<run_name>/chesscog_vs_ours.csv`

Milestone: M4.

**TODO** (see `docs/M4_TODO.md`)
- [ ] `run_chesscog`: install in a separate venv, check it works on warped frames or needs
      raw frames, write the `fen_chesscog` CSV
- [ ] `compare`, `squares_wrong`, `_placement`
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import chess
import numpy as np

## Running chesscog

In [ ]:
def run_chesscog(
    image_paths: Sequence[str | Path],
    corner_hint: np.ndarray | None = None,
    turn: chess.Color = chess.WHITE,
) -> list[str]:
    """Run chesscog on full-board images and return one board_fen per image.

    Install (separate venv recommended; chesscog pins its own torch/torchvision versions):
        pip install chesscog
        python -m chesscog.occupancy_classifier.download_model
        python -m chesscog.piece_classifier.download_model

    Invocation (verify against the repo, chesscog/recognition/recognition.py):
        chesscog.recognition.recognition.ChessRecognizer().predict(rgb_image, turn)
        -> (chess.Board, corners (4, 2)).
    CLI equivalent: `python -m chesscog.recognition.recognition <image> --white`.
    """
    # Design / open questions:
    # - chesscog finds the board corners itself from a perspective photo. Our frames/ images
    #   are already warped top-down; check whether its corner detection works on them, or
    #   feed it raw camera frames instead (then save raw frames in M2 capture).
    # - It expects RGB (cv2.imread gives BGR).
    # - corner_hint: chesscog has no public corner-input API as far as known; if detection
    #   fails, consider bypassing its corner step with our ArUco corners. Unverified.
    # - `turn` sets the board orientation chesscog assumes; match our camera placement.
    # - Record failures (exception / no board found) as an empty string, not a crash.
    # TODO: implement
    raise NotImplementedError

## Comparing
A missing or unparsable prediction counts as all 64 squares wrong.

In [ ]:
def compare(ours_csv: str | Path, chesscog_csv: str | Path, out_csv: str | Path):
    """Join our per-board predictions with chesscog's and write chesscog_vs_ours.csv.

    ours_csv columns: session_id, frame_id, fen_true, fen_ours
    chesscog_csv columns: session_id, frame_id, fen_chesscog
    Output adds: correct_ours, correct_chesscog, squares_wrong_ours, squares_wrong_chesscog.
    Returns the merged DataFrame.
    """
    # Design:
    # - Read both CSVs as str with keep_default_na=False (frame ids like "000001" stay
    #   strings; empty FENs stay "", not NaN).
    # - Left join on (session_id, frame_id): every board of ours appears, missing chesscog
    #   predictions become "".
    # - Per method: correct = placements equal; squares_wrong via squares_wrong().
    # TODO: implement
    raise NotImplementedError


def squares_wrong(true_fen: str, pred_fen: str) -> int:
    """Number of the 64 squares that differ; 64 if the prediction is missing or invalid."""
    # Design: parse both placements into empty boards (Board(None).set_board_fen), ValueError
    # -> 64, otherwise count squares where piece_at differs.
    # TODO: implement
    raise NotImplementedError


def _placement(fen: str) -> str:
    """Placement field of a FEN ("" for a blank string)."""
    # TODO: implement
    raise NotImplementedError

## Self-check
(was the chesscog half of tests/test_evaluate.py)

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - squares_wrong(STARTING_FEN, board after 1. e4) == 2; squares_wrong(STARTING_FEN, "") == 64.
# - compare on two tiny CSVs in a tempdir (ours correct = start position, chesscog = after
#   1. e4): correct_ours True, correct_chesscog False, squares_wrong_chesscog 2, output file exists.
# - run_chesscog on one saved frame returns a valid board_fen (needs the chesscog install,
#   so keep it out of this notebook's self-check; test it in run_chesscog.ipynb).
print("chesscog_compare: self-check not written yet (see plan above)")